# Performance

## Measurement Scope

This page generates charts and supporting tables at book compilation from the document-reported
aggregates in `milestone1/data/reported_results.json`, reviewed on
**9 November 2026**. Hieu's document supplies arXiv Fast Crawl/Recovery time
and compressed checkpoint sizes; Dat's document supplies cache-heavy Semantic
Scholar rerun timings and notebook-process RSS. Original run exports and exact
per-run configurations are pending reconciliation.

The technical documents report Google Colab CPU-only execution. The supplied
arXiv notebooks configure three workers, while the arXiv document describes a
single active connection; the actual benchmark configuration needs
confirmation. These separate measurements do not yet constitute a complete
end-to-end M1 benchmark.

In [ ]:
from pathlib import Path
import sys

# Support execution from the project root or the notebook's own directory.
root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "myst.yml").is_file() and (p / "milestone1" / "reporting.py").is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from milestone1.reporting import load_results, total, integer, percent, seconds, hms, table

data = load_results()
arxiv = data["arxiv"]["ranges"]
s2 = data["semantic_scholar"]["ranges"]

from milestone1.reporting import COLORS, chart_style, finish_figure
from matplotlib.ticker import FuncFormatter

plt = chart_style()
labels = [f"Range {r['range']}" for r in arxiv]
positions = list(range(len(labels)))
comma = FuncFormatter(lambda value, _: f"{value:,.0f}")

## arXiv Recorded Crawl and Recovery Time

**Figure 1.** Logged Fast Crawl and Recovery durations, in hours. Stacked bar
lengths are computed from the two printed phase durations; labels above the
bars are their sums, rounded to two decimals. They are recorded invocation
time, not necessarily calendar time including gaps between Colab sessions.
Small differences against the document's printed totals are audited below.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8), layout="constrained")
fast = [seconds(r["fast_hms"])/3600 for r in arxiv]
recovery = [seconds(r["recovery_hms"])/3600 for r in arxiv]
ax.bar(positions, fast, color=COLORS["blue"], label="Fast Crawl")
ax.bar(positions, recovery, bottom=fast, color=COLORS["orange"], label="Recovery", hatch="//")
for i, (f, r) in enumerate(zip(fast, recovery)):
    ax.text(i, f/2, f"{f:.2f}", ha="center", va="center", color="white", fontweight="bold")
    ax.text(i, f+r/2, f"{r:.2f}", ha="center", va="center")
    ax.text(i, f+r+0.35, f"{f+r:.2f} h", ha="center", fontsize=10)
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 22),
       ylabel="Recorded invocation time (hours)", title="arXiv: Fast Crawl and Recovery")
ax.grid(axis="y", alpha=0.18)
ax.legend(loc="upper left", frameon=False)
finish_figure(fig, "performance-arxiv-runtime",
              "Recorded arXiv Fast Crawl and Recovery hours by range; Recovery is largest in Range 5.")

The document reports **61:41:45** of recorded invocation time (about 61.70
hours), or 8.884 seconds per assigned ID. Recovery accounts for approximately
52.1%. Range totals vary from 7:39:30 to 18:41:33, with 5.514--13.459 seconds
per assigned ID. Fast Crawl averages approximately 3.653--4.782 seconds per
assigned ID across the five ranges.

Per-ID time is not per-request latency: metadata batching and multiple source
versions change the number of HTTP requests. It cannot independently establish
request-rate compliance. The final error count also cannot quantify recovery
effectiveness without the pre-Recovery failures and attempted recoveries.
Possible effects of temporary failures and waiting remain hypotheses until
logs are reconciled.

No 18.7-hour parallel completion time is claimed: taking the slowest range
does not establish a measured group speedup, and shared access constraints
must still be respected. Runtime ledgers omit older untracked runs and sessions
terminated before a ledger event was written.


Range 5 records the longest Recovery duration (12:04:16), while Range 4
records the shortest (1:57:31). Recovery exceeds Fast Crawl in Ranges 2, 3
and 5. These comparisons describe where recorded time was spent; they do not
explain the underlying causes without retry and recovery logs.

## Compressed Checkpoint Storage

**Figure 2.** Final retained checkpoint archive sizes, in MiB, including
retained sources and bookkeeping. This measures compressed output, not peak
disk usage or the uncompressed size of successful source versions.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.4), layout="constrained")
sizes = [r["checkpoint_mib"] for r in arxiv]
bars = ax.bar(positions, sizes, color=COLORS["teal"])
ax.bar_label(bars, labels=[f"{v:,.2f}" for v in sizes], padding=4)
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 350),
       ylabel="Compressed checkpoint storage (MiB)", title="arXiv: retained checkpoint storage")
ax.grid(axis="y", alpha=0.18)
finish_figure(fig, "performance-checkpoint-storage",
              "Final compressed arXiv checkpoint sizes range from 237.15 to 291.45 MiB.")

The reported retained checkpoint total is **1,360.67 MiB**, approximately
1.33 GiB and 55.7 KiB per assigned ID. Range sizes are 237.15--291.45 MiB.
This is final compressed output storage, not peak disk use during execution.
It does not quantify the storage reduction caused by removing figures without
corresponding before/after source sizes and stated compression conventions.

The code records batch RSS, scratch size, discovery time and source-size
fields, but those values were not supplied in the arXiv document. Existing
`metrics.csv` and checkpoint metrics should be checked before deciding whether
additional measurements are needed. Latest-batch metrics can describe Recovery
rather than the initial crawl, and sampling stops before checkpoint packaging.


Range 3 has the largest retained archives and Range 1 the smallest, despite
equal assigned ID counts. Version counts and source composition may differ,
but the supplied aggregates do not support attributing the storage difference
to a particular cause.

## Semantic Scholar Cache-Heavy Rerun

### Timing scopes

**Figure 3.** Fetch timer totals and notebook-cell wall times are shown in
separate panels with different vertical scales. Labels below the range names
give fresh-publication lookup counts. These are cache-heavy reruns, **not
full fresh crawls**. One fresh lookup may contain several HTTP attempts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), layout="constrained")
fresh = [r["input_papers"]-r["cache_hits"]-r["invalid_id"] for r in s2]
tick_labels = [f"Range {r['range']}\n{n} fresh" for r,n in zip(s2,fresh)]
for ax, key, title, color in (
    (axes[0], "fetch_seconds", "Accumulated fresh-lookup fetch time", COLORS["blue"]),
    (axes[1], "cell_seconds", "Notebook-cell wall time", COLORS["purple"]),
):
    values = [r[key] for r in s2]
    bars = ax.bar(positions, values, color=color)
    ax.bar_label(bars, labels=[f"{v:,.2f}" for v in values], padding=4, fontsize=9)
    ax.set(xticks=positions, xticklabels=tick_labels, ylim=(0, max(values)*1.2),
           ylabel="Time (seconds)", title=title)
    ax.grid(axis="y", alpha=0.18)
finish_figure(fig, "performance-semantic-scholar-timing",
              "Cache-heavy Semantic Scholar rerun timings with separate scales for fetch and cell time, and fresh lookup counts.")

Across the five reported reruns, **24,811 of 25,000** inputs are cache hits,
leaving **189** fresh valid-publication lookups. The displayed fetch times do
not represent retrieving all 25,000 publications from the API without cache.
Neither 189 nor the number of papers is an internal HTTP-attempt counter.

Range 4 reports 1.35 seconds of fetch time but 1,411.08 seconds of cell time.
The measurements have different scopes; the supplied aggregates do not
establish whether cache I/O, other local processing or another cause explains
the difference. No causal diagnosis or inter-range API-speed ranking is made.


Per-paper fetch timing ends before normalization/cache writes. The cell timer
includes its local processing but excludes earlier checkpoint loading and later
table construction/validation. The reported mean fetch times per fresh
publication range from 0.271 to 1.961 seconds; they are not per-HTTP-attempt
latencies and cannot rank full-corpus crawl performance.

### Notebook-process memory

**Figure 4.** Average and peak sampled notebook-process RSS for each reported
rerun. Both series start at zero and share the same MiB scale. Their near-equal
values are preserved; differences are not exaggerated with a truncated axis.
These measurements include the process's existing memory.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8), layout="constrained")
width = 0.36
averages = [r["average_rss_mib"] for r in s2]
peaks = [r["peak_rss_mib"] for r in s2]
avg_bars = ax.bar([i-width/2 for i in positions], averages, width,
                  color=COLORS["blue"], label="Average sampled RSS")
peak_bars = ax.bar([i+width/2 for i in positions], peaks, width,
                   color=COLORS["orange"], label="Peak sampled RSS", hatch="//")
for bars, values in ((avg_bars, averages), (peak_bars, peaks)):
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in values], padding=4, fontsize=9, rotation=90)
ax.set(xticks=positions, xticklabels=labels, ylim=(0, 350),
       ylabel="Notebook-process RSS (MiB)", title="Semantic Scholar: cache-heavy rerun memory")
ax.grid(axis="y", alpha=0.18)
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center", ncol=2, frameon=False)
finish_figure(fig, "performance-semantic-scholar-memory",
              "Average and peak notebook-process RSS on a zero-based axis; peaks range from 246.29 to 269.21 MiB.")

`track_run` samples RSS every 0.5 seconds. The reported average and peak values
differ by at most 0.03 MiB within a range. This snapshot shows little sampled
variation, but it does not establish that memory remains stable during a fresh
crawl. Reported peaks range from 246.29 to 269.21 MiB; these are observed sampled
maxima, not full-system M1 peaks. Range means are not pooled into an unweighted
overall RAM average.

Metrics/CSV are overwritten per rerun; unlike the arXiv runtime ledger, they
are not a cumulative history. Zero final error flags do not establish zero
retries or HTTP 429 responses. Corpus size and peak disk were not supplied.

## Supporting Audit and Pending Measurements

### Timing arithmetic

**Table 1.** The source document's displayed arXiv times contain small
discrepancies. Reported totals are retained here alongside recomputed values;
the chart uses the printed phase durations. Reconcile against unrounded ledger
seconds before finalizing the benchmark. The reported overall total of
61:41:45 corresponds to 8.884 seconds per assigned ID.

In [ ]:
overall = data["arxiv"]["reported_overall"]
audit = []
for r in arxiv:
    calculated = seconds(r["fast_hms"]) + seconds(r["recovery_hms"])
    reported = seconds(r["total_hms"])
    if calculated != reported:
        audit.append([f"Range {r['range']}: phase sum", r["total_hms"], hms(calculated), f"{calculated - reported:+d}"])
for label, key in (("Overall Fast Crawl: row sum", "fast_hms"),
                   ("Overall Recovery: row sum", "recovery_hms"),
                   ("Overall total: row sum", "total_hms")):
    calculated = sum(seconds(r[key]) for r in arxiv)
    reported = seconds(overall[key])
    audit.append([label, overall[key], hms(calculated), f"{calculated - reported:+d}"])
phase_sum = sum(seconds(r["fast_hms"]) + seconds(r["recovery_hms"]) for r in arxiv)
audit.append(["Overall: sum of all printed phases", overall["total_hms"], hms(phase_sum),
              f"{phase_sum - seconds(overall['total_hms']):+d}"])
table(["Check", "Reported", "Recomputed from printed values", "Difference (s)"], audit)

### Evidence still needed

**Table 2.** These gaps prevent the separate stage measurements from being
presented as a complete end-to-end M1 benchmark. Missing metrics are not
plotted as zero or filled with estimates.

In [ ]:
table(["Measurement", "Current evidence", "Next evidence needed"], [
    ["arXiv time", "Document-reported logged invocations", "Raw runtime ledgers; rounding and configurations"],
    ["arXiv RAM / discovery / source sizes", "Measurement fields exist in code; values not supplied", "Per-batch metrics and their run scope"],
    ["arXiv disk", "Final compressed checkpoint sizes", "Peak full-system storage, including temporary/backup use"],
    ["Recovery effectiveness", "Final 22 publication errors", "Pre-Recovery counts and version/reason history"],
    ["Semantic Scholar time", "Cache-heavy rerun cell/fetch timers", "Original fresh-crawl logs or a clearly scoped benchmark"],
    ["Semantic Scholar retries / disk", "No independent aggregate supplied", "Recorded retry/status counters and storage measurement"],
    ["Full M1 benchmark", "Separate stage measurements", "Auditable CPU-only end-to-end wall time and resource footprint"],
])

The two stages' unrelated run timings are not added to invent an end-to-end
measurement. Once the owners provide the original exports or scoped additional
measurements, this page can be updated without changing the report URL.